## 1. Preparar o ambiente e localizar os arquivos

Definimos os caminhos (mesmos do `ETL.ipynb`) e listamos os seis CSVs semestrais da ANP.

In [2]:
import csv
import getpass
import glob
import os

from sqlalchemy import URL, create_engine, text

PASTA_BRUTOS = os.path.join("..", "dados", "sujo")
arquivos_csv = sorted(glob.glob(os.path.join(PASTA_BRUTOS, "ca-*.csv")))

assert len(arquivos_csv) == 6, f"Esperava 6 arquivos, encontrei {len(arquivos_csv)}"
print(f"Arquivos encontrados ({len(arquivos_csv)}):")
for caminho in arquivos_csv:
    print(f"  {os.path.basename(caminho)}  ({os.path.getsize(caminho) / 1e6:,.1f} MB)")

Arquivos encontrados (6):
  ca-2018-01.csv  (82.4 MB)
  ca-2018-02.csv  (86.6 MB)
  ca-2019-01.csv  (87.2 MB)
  ca-2019-02.csv  (89.0 MB)
  ca-2020-01.csv  (86.7 MB)
  ca-2020-02.csv  (38.9 MB)


## 2. Conectar ao PostgreSQL

Configure `PGHOST`, `PGDATABASE` e `PGUSER` no ambiente antes de executar (`PGPORT` é opcional e usa 5432). A senha vem de `PGPASSWORD`; se não existir, o notebook pede sem exibi-la. **Não salve credenciais no Git.**

In [ ]:
configuracoes_obrigatorias = ("PGHOST", "PGDATABASE", "PGUSER")
faltando = [nome for nome in configuracoes_obrigatorias if not os.getenv(nome)]
if faltando:
    raise RuntimeError(f"Configure as variáveis de ambiente: {', '.join(faltando)}")

senha = os.getenv("PGPASSWORD") or getpass.getpass("Senha do PostgreSQL: ")
url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["PGUSER"],
    password=senha,
    host=os.environ["PGHOST"],
    port=int(os.getenv("PGPORT", "5432")),
    database=os.environ["PGDATABASE"],
)
engine = create_engine(url, pool_pre_ping=True)

with engine.connect() as conexao:
    banco = conexao.execute(text("SELECT current_database()")).scalar_one()
print(f"Conectado ao banco: {banco}")

## 3. Definir as tabelas sujas

Criamos uma tabela por arquivo (`ca_2018_01` … `ca_2020_02`) no schema `raw_anp`.

Decisões desta camada:

* **Todas as colunas são `TEXT`.** Nenhum valor é interpretado: o CNPJ continua com pontos e espaço inicial, o preço continua com vírgula e a data continua como `dd/mm/aaaa`. A conversão de tipos é feita na etapa **T**.
* **Os nomes das colunas viram `snake_case` apenas no DDL**, porque nomes como `"Regiao - Sigla"` são ruins de usar em SQL. O conteúdo não é alterado.
* **A carga é refeita do zero a cada execução** (`DROP TABLE IF EXISTS`), então rodar o notebook duas vezes não duplica dados.

In [ ]:
SCHEMA_RAW = "raw_anp"

# Ordem idêntica às 16 colunas dos CSVs da ANP.
COLUNAS = [
    "regiao_sigla", "estado_sigla", "municipio", "revenda", "cnpj_revenda",
    "nome_rua", "numero_rua", "complemento", "bairro", "cep", "produto",
    "data_coleta", "valor_venda", "valor_compra", "unidade_medida", "bandeira",
]


def nome_tabela(caminho: str) -> str:
    """ca-2018-01.csv -> ca_2018_01"""
    base = os.path.splitext(os.path.basename(caminho))[0]
    return base.replace("-", "_")


TABELAS = {caminho: nome_tabela(caminho) for caminho in arquivos_csv}

def ddl_tabela(tabela: str) -> str:
    colunas_sql = ",\n    ".join(f"{c} TEXT" for c in COLUNAS)
    return f"CREATE TABLE {SCHEMA_RAW}.{tabela} (\n    {colunas_sql}\n)"

print(ddl_tabela("ca_2018_01"))

## 4. Conferir o cabeçalho de cada CSV antes de carregar

O `COPY` mapeia as colunas **por posição** e ignora a linha de cabeçalho. Por isso, antes de carregar, confirmamos que todos os arquivos têm exatamente 16 colunas e o mesmo cabeçalho. Se algum arquivo estiver diferente, a carga é interrompida.

In [ ]:
def ler_cabecalho(caminho: str) -> list[str]:
    with open(caminho, encoding="utf-8-sig", newline="") as f:  # utf-8-sig remove o BOM
        return next(csv.reader(f, delimiter=";"))

cabecalhos = {os.path.basename(c): ler_cabecalho(c) for c in arquivos_csv}
referencia = next(iter(cabecalhos.values()))

for nome, cab in cabecalhos.items():
    assert len(cab) == len(COLUNAS), f"{nome}: {len(cab)} colunas (esperado {len(COLUNAS)})"
    assert cab == referencia, f"{nome}: cabeçalho diferente dos demais arquivos"

print("Cabeçalho único em todos os arquivos:")
print(referencia)

## 5. Extração + Carga (EL)

São cerca de 2,7 milhões de linhas, então usamos o `COPY` nativo do PostgreSQL (via `psycopg2`), que é muito mais rápido que `DataFrame.to_sql`. O arquivo é enviado direto ao servidor, sem passar pelo Pandas e **sem descartar nenhuma linha** (diferente de `on_bad_lines='skip'`, que perderia registros em silêncio).

Cada arquivo é carregado em sua própria transação: se um falhar, aquela tabela é desfeita e o erro aparece.

In [ ]:
import time

conn = engine.raw_connection()          # conexão psycopg2 "crua", necessária para o COPY
try:
    with conn.cursor() as cur:
        cur.execute(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_RAW}")
    conn.commit()

    for caminho, tabela in TABELAS.items():
        inicio = time.time()
        try:
            with conn.cursor() as cur:
                cur.execute(f"DROP TABLE IF EXISTS {SCHEMA_RAW}.{tabela}")
                cur.execute(ddl_tabela(tabela))
                with open(caminho, encoding="utf-8-sig", newline="") as f:
                    cur.copy_expert(
                        f"COPY {SCHEMA_RAW}.{tabela} FROM STDIN "
                        "WITH (FORMAT csv, HEADER true, DELIMITER ';')",
                        f,
                    )
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        print(f"{SCHEMA_RAW}.{tabela}: carregada em {time.time() - inicio:,.0f}s")
finally:
    conn.close()

print("\nProcesso de EL (Extração e Carga) concluído.")

## 6. Provar que nada se perdeu na carga

Contamos os registros **direto no arquivo** (com o módulo `csv`, que respeita campos entre aspas) e comparamos com a contagem **no banco**. As duas precisam ser iguais em todos os semestres.

Por fim, comparamos o total com o do `ETL.ipynb` (**2.686.331** registros): se os dois pipelines partem da mesma base, o número tem que bater.

In [ ]:
def contar_registros_arquivo(caminho: str) -> int:
    with open(caminho, encoding="utf-8-sig", newline="") as f:
        return sum(1 for _ in csv.reader(f, delimiter=";")) - 1   # tira o cabeçalho

TOTAL_ETL = 2_686_331   # total lido pelo ETL.ipynb

resultados = []
with engine.connect() as conexao:
    for caminho, tabela in TABELAS.items():
        no_arquivo = contar_registros_arquivo(caminho)
        no_banco = conexao.execute(text(f"SELECT COUNT(*) FROM {SCHEMA_RAW}.{tabela}")).scalar_one()
        resultados.append((tabela, no_arquivo, no_banco))
        status = "OK" if no_arquivo == no_banco else "DIVERGE"
        print(f"{SCHEMA_RAW}.{tabela}: arquivo={no_arquivo:>9,}  banco={no_banco:>9,}  [{status}]")

divergentes = [t for t, a, b in resultados if a != b]
assert not divergentes, f"Tabelas com perda ou excesso de linhas: {divergentes}"

total_banco = sum(b for _, _, b in resultados)
print(f"\nTotal no banco: {total_banco:,}")
print(f"Total no ETL:   {TOTAL_ETL:,}  ->  " + ("bate com o ETL" if total_banco == TOTAL_ETL else "NÃO bate com o ETL"))

## 7. Conhecer a camada suja

Como nada foi limpo, aqui vemos os problemas reais da fonte que a etapa **T** vai resolver: campos vazios, espaços no CNPJ, preço com vírgula, variações de bandeira. Também conferimos a presença de `valor_compra` em cada semestre, informação que o README original dá como ausente em parte de 2020.

In [ ]:
with engine.connect() as conexao:
    print("Preenchimento por semestre (linhas / com valor_compra / com complemento):")
    for tabela in TABELAS.values():
        linhas, compra, compl = conexao.execute(text(
            f"SELECT COUNT(*), COUNT(NULLIF(TRIM(valor_compra), '')), "
            f"COUNT(NULLIF(TRIM(complemento), '')) FROM {SCHEMA_RAW}.{tabela}"
        )).one()
        print(f"  {tabela}: {linhas:>9,} / {compra:>9,} / {compl:>9,}")

    print("\nExemplo de linhas brutas (observe espaço no CNPJ, vírgula no preço e data dd/mm/aaaa):")
    amostra = conexao.execute(text(
        f"SELECT cnpj_revenda, produto, data_coleta, valor_venda, valor_compra, bandeira "
        f"FROM {SCHEMA_RAW}.ca_2018_01 LIMIT 3"
    )).all()
    for linha in amostra:
        print("  ", [repr(v) for v in linha])

## 8. Transformação (T) com dbt

A partir daqui a limpeza **não usa mais Python**: o dbt lê as tabelas de `raw_anp` e constrói o Data Warehouse no schema `dw_elt`, com as mesmas tabelas do ETL (`dim_tempo`, `dim_posto`, `dim_localizacao`, `dim_produto`, `dim_bandeira` e `fato_preco`).

Camadas previstas no projeto dbt:

1. **staging** — `UNION ALL` das seis tabelas sujas, acrescentando `arquivo_origem`;
2. **intermediate** — limpeza em SQL (CNPJ/CEP só com dígitos, `S/N`, data, preço, bandeira, outliers);
3. **marts** — dimensões com chaves substitutas e a fato.

A célula abaixo só executa o dbt se a pasta do projeto existir; caso contrário, avisa o que falta.

In [ ]:
import subprocess
from pathlib import Path

PROJETO_DBT = Path("..") / "transformacao_combustiveis"

if not (PROJETO_DBT / "dbt_project.yml").is_file():
    print(f"Projeto dbt ainda não criado em {PROJETO_DBT.resolve()}")
    print("Crie-o com `dbt init transformacao_combustiveis` e construa os modelos; depois rode esta célula.")
else:
    for comando in (["dbt", "debug"], ["dbt", "build"]):
        print("$", " ".join(comando))
        resultado = subprocess.run(comando, cwd=PROJETO_DBT, capture_output=True, text=True)
        print(resultado.stdout[-2000:])
        if resultado.returncode != 0:
            print(resultado.stderr[-2000:])
            raise RuntimeError(f"Falha em: {' '.join(comando)}")

## 9. Comparar ELT (`dw_elt`) × ETL (`dw_etl`)

Se os dois caminhos estão corretos, as tabelas finais devem ter o **mesmo conteúdo**. Comparamos as contagens de cada tabela e alguns agregados da fato. A célula só compara o que existir em cada schema.

In [ ]:
TABELAS_DW = ["dim_tempo", "dim_posto", "dim_localizacao", "dim_produto", "dim_bandeira", "fato_preco"]

def existe(conexao, schema: str, tabela: str) -> bool:
    return conexao.execute(
        text("SELECT to_regclass(:nome) IS NOT NULL"), {"nome": f"{schema}.{tabela}"}
    ).scalar_one()

with engine.connect() as conexao:
    print(f"{'tabela':<18}{'dw_etl':>12}{'dw_elt':>12}   resultado")
    for tabela in TABELAS_DW:
        n = {}
        for schema in ("dw_etl", "dw_elt"):
            n[schema] = (
                conexao.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}")).scalar_one()
                if existe(conexao, schema, tabela) else None
            )
        if None in n.values():
            resultado = "aguardando " + ", ".join(s for s, v in n.items() if v is None)
        else:
            resultado = "igual" if n["dw_etl"] == n["dw_elt"] else "DIFERENTE"
        fmt = lambda v: f"{v:,}" if v is not None else "-"
        print(f"{tabela:<18}{fmt(n['dw_etl']):>12}{fmt(n['dw_elt']):>12}   {resultado}")

    if existe(conexao, "dw_etl", "fato_preco") and existe(conexao, "dw_elt", "fato_preco"):
        print("\nAgregados da fato (soma de valor_venda e total de outliers):")
        for schema in ("dw_etl", "dw_elt"):
            soma, outliers = conexao.execute(text(
                f"SELECT ROUND(SUM(valor_venda), 2), COUNT(*) FILTER (WHERE fl_outlier) FROM {schema}.fato_preco"
            )).one()
            print(f"  {schema}: soma={soma}  outliers={outliers:,}")

## 10. Encerrar

In [ ]:
engine.dispose()
print("Conexão encerrada.")